# Deteksi Anomali Pasar Saham Berbasis Hybrid Time Series Decomposition dan Machine Learning dengan Interpretasi SHAP sebagai Early Warning System**Data Analysis Competition — Matematika Fair 2026****Tema:** Peran Matematika Kritis dalam Membangun Inovasi dan Solusi di Era Digital

## 1. Setup & Import Libraries**Justifikasi Matematis:**Proyek ini berlandaskan pada prinsip *Matematika Kritis* yang memandang bahwa pasar keuangan bukanlah sistem yang sepenuhnya efisien (*Efficient Market Hypothesis*). Secara matematis, return saham seringkali melanggar asumsi distribusi normal — menunjukkan *leptokurtosis* (fat-tails) dan *volatility clustering*. Fakta ini memotivasi penggunaan pendekatan hybrid: dekomposisi time series untuk mengekstrak struktur laten, dikombinasikan dengan machine learning untuk menangkap pola nonlinear yang tidak tertangkap oleh model parametrik klasik.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom statsmodels.tsa.seasonal import STLfrom statsmodels.stats.stattools import durbin_watsonfrom sklearn.ensemble import IsolationForestfrom sklearn.preprocessing import StandardScalerimport shapimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')plt.rcParams['figure.dpi'] = 120

## 2. Data Loading & Preprocessing**Justifikasi Matematis:**Log-return didefinisikan sebagai  = \ln(P_t / P_{t-1})$, di mana $ adalah harga penutupan pada waktu $. Pemilihan log-return dibandingkan simple return memiliki keunggulan matematis: (1) bersifat *time-additive* sehingga return multi-periode cukup dijumlahkan, dan (2) untuk nilai return kecil, $\ln(1+r) pprox r$, namun lebih stabil secara numerik pada perubahan harga ekstrem. Pemilihan saham BBCA sebagai prototipe didasarkan pada likuiditas tinggi dan representativitas terhadap sektor perbankan Indonesia.

In [ ]:
df_raw = pd.read_csv('Dataset_DAC_.csv', sep=';')df_raw['Stock_Name'] = df_raw['Stock_Name'].str.strip()df_raw['Date'] = pd.to_datetime(df_raw['Date'], format='%d/%m/%Y %H:%M')print(f"Total records: {df_raw.shape[0]}")print(f"Unique stocks: {df_raw['Stock_Name'].nunique()}")print(f"Date range: {df_raw['Date'].min()} — {df_raw['Date'].max()}")df_raw.head()

In [ ]:
df = df_raw[df_raw['Stock_Name'] == 'BBCA'].copy()df = df.sort_values('Date').reset_index(drop=True)df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))df = df.dropna(subset=['Log_Return']).reset_index(drop=True)print(f"BBCA records: {df.shape[0]}")print(f"Period: {df['Date'].iloc[0].date()} — {df['Date'].iloc[-1].date()}")df[['Date', 'Close', 'Log_Return']].head(10)

## 3. Uji Asumsi Statistik**Justifikasi Matematis:**Dua asumsi fundamental yang sering diasumsikan dalam model keuangan klasik perlu diuji secara empiris:1. **Uji Kolmogorov-Smirnov (KS):** Menguji hipotesis $: return saham berdistribusi normal. Statistik KS didefinisikan sebagai  = \sup_x |F_n(x) - F(x)|$, yakni supremum jarak antara fungsi distribusi empiris (x)$ dan fungsi distribusi normal teoretis (x)$. Penolakan $ mengkonfirmasi keberadaan *fat-tails* (kelebihan kurtosis), yang merupakan dasar mengapa model berbasis asumsi Gaussian tidak memadai untuk deteksi anomali pasar.2. **Uji Durbin-Watson (DW):** Menguji autokorelasi orde pertama pada residual. Statistik DW didefinisikan sebagai  = rac{\sum_{t=2}^{n}(e_t - e_{t-1})^2}{\sum_{t=1}^{n}e_t^2}$, dengan nilai  pprox 2$ mengindikasikan tidak ada autokorelasi. Deviasi signifikan dari 2 mengindikasikan adanya *memory effect* dalam pasar — bahwa return masa lalu mempengaruhi return masa depan, berlawanan dengan asumsi *random walk*.

In [ ]:
print("=" * 60)print("UJI ASUMSI STATISTIK — LOG RETURN BBCA")print("=" * 60)returns = df['Log_Return'].valuesks_stat, ks_pval = stats.kstest(returns, 'norm', args=(returns.mean(), returns.std()))print(f"[1] Kolmogorov-Smirnov Test")print(f"    H0: Log-return berdistribusi normal")print(f"    Statistik KS  : {ks_stat:.6f}")print(f"    p-value        : {ks_pval:.2e}")print(f"    Kurtosis       : {stats.kurtosis(returns, fisher=True):.4f}")print(f"    Skewness       : {stats.skew(returns):.4f}")if ks_pval < 0.05:    print(f"    Kesimpulan     : Tolak H0 (α=0.05). Return TIDAK berdistribusi normal.")    print(f"                     Terdapat fat-tails → model Gaussian tidak memadai.")else:    print(f"    Kesimpulan     : Gagal tolak H0. Return mendekati distribusi normal.")dw_stat = durbin_watson(returns)print(f"[2] Durbin-Watson Test")print(f"    H0: Tidak ada autokorelasi orde pertama")print(f"    Statistik DW   : {dw_stat:.6f}")if dw_stat < 1.5:    print(f"    Kesimpulan     : Terdapat autokorelasi positif (DW < 1.5).")    print(f"                     Pasar memiliki memory effect.")elif dw_stat > 2.5:    print(f"    Kesimpulan     : Terdapat autokorelasi negatif (DW > 2.5).")else:    print(f"    Kesimpulan     : Tidak ada bukti kuat autokorelasi (DW ≈ 2).")print("" + "=" * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].hist(returns, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')x = np.linspace(returns.min(), returns.max(), 300)axes[0].plot(x, stats.norm.pdf(x, returns.mean(), returns.std()), 'r-', lw=2, label='Normal fit')axes[0].set_title('Distribusi Log-Return BBCA vs Normal')axes[0].set_xlabel('Log-Return')axes[0].set_ylabel('Density')axes[0].legend()stats.probplot(returns, dist='norm', plot=axes[1])axes[1].set_title('Q-Q Plot Log-Return BBCA')plt.tight_layout()plt.show()

## 4. Time Series Decomposition & Feature Engineering**Justifikasi Matematis:**Dekomposisi STL (*Seasonal and Trend decomposition using Loess*) memecah deret waktu harga $ menjadi tiga komponen aditif:86P_t = T_t + S_t + R_t86di mana $ adalah komponen tren (pergerakan jangka panjang), $ adalah komponen musiman (pola berulang), dan $ adalah residual (deviasi tak terjelaskan). Komponen $ inilah yang menjadi sinyal utama deteksi anomali — lonjakan residual mengindikasikan peristiwa yang menyimpang dari pola historis normal.Fitur tambahan yang diekstrak:- **Rolling Volatility 30-hari:** $\sigma_{30,t} = 	ext{std}(r_{t-29}, \ldots, r_t)$ — mengukur dispersi return dalam jendela 30 hari, menangkap fenomena *volatility clustering* (Mandelbrot, 1963).- **Volume Ratio:**  = V_t / ar{V}_{30,t}$ — rasio volume perdagangan harian terhadap rata-rata 30 hari, mendeteksi lonjakan aktivitas yang sering menyertai peristiwa anomali.

In [ ]:
stl = STL(df['Close'], period=252, robust=True)result = stl.fit()df['Trend'] = result.trenddf['Seasonal'] = result.seasonaldf['Residual'] = result.resid

In [ ]:
df['Rolling_Vol_30'] = df['Log_Return'].rolling(window=30).std()df['Volume_Ratio'] = df['Volume'] / df['Volume'].rolling(window=30).mean()df_model = df.dropna(subset=['Residual', 'Rolling_Vol_30', 'Volume_Ratio']).copy()df_model = df_model.reset_index(drop=True)print(f"Records setelah feature engineering: {df_model.shape[0]}")df_model[['Date', 'Close', 'Residual', 'Rolling_Vol_30', 'Volume_Ratio']].describe()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)axes[0].plot(df_model['Date'], df_model['Close'], color='steelblue', lw=0.8)axes[0].set_title('Harga Close BBCA')axes[0].set_ylabel('Harga (IDR)')axes[1].plot(df_model['Date'], df_model['Residual'], color='darkorange', lw=0.6)axes[1].set_title('STL Residual')axes[1].set_ylabel('Residual')axes[2].plot(df_model['Date'], df_model['Rolling_Vol_30'], color='crimson', lw=0.8)axes[2].set_title('Rolling Volatility 30-hari')axes[2].set_ylabel('Volatilitas')axes[3].plot(df_model['Date'], df_model['Volume_Ratio'], color='seagreen', lw=0.6)axes[3].set_title('Volume Ratio (V_t / V̄_30)')axes[3].set_ylabel('Rasio')axes[3].set_xlabel('Tanggal')plt.tight_layout()plt.show()

## 5. Anomaly Detection — Isolation Forest**Justifikasi Matematis:**Isolation Forest bekerja berdasarkan prinsip bahwa anomali adalah observasi yang *mudah diisolasi*. Algoritma membangun ensemble dari *isolation trees* dengan melakukan partisi rekursif secara acak. Untuk setiap observasi $, *anomaly score* didefinisikan sebagai:86s(x_i, n) = 2^{-rac{E[h(x_i)]}{c(n)}}86di mana [h(x_i)]$ adalah rata-rata *path length* (jumlah partisi yang dibutuhkan untuk mengisolasi $) dari seluruh trees, dan (n)$ adalah faktor normalisasi berdasarkan rata-rata *path length* pada Binary Search Tree dengan $ observasi. Skor mendekati 1 mengindikasikan anomali kuat (mudah diisolasi), sedangkan skor mendekati 0.5 mengindikasikan observasi normal.Parameter  mengasumsikan bahwa sekitar 2% dari seluruh periode perdagangan merupakan hari krisis/anomali — sejalan dengan *empirical rule* bahwa peristiwa ekstrem di pasar keuangan terjadi pada ekor distribusi.

In [ ]:
features = ['Residual', 'Rolling_Vol_30', 'Volume_Ratio']scaler = StandardScaler()X_scaled = scaler.fit_transform(df_model[features])iso_forest = IsolationForest(    n_estimators=300,    contamination=0.02,    max_samples='auto',    random_state=42,    n_jobs=-1)iso_forest.fit(X_scaled)df_model['Anomaly'] = iso_forest.predict(X_scaled)df_model['Anomaly_Score'] = iso_forest.decision_function(X_scaled)df_model['Anomaly_Label'] = df_model['Anomaly'].map({1: 'Normal', -1: 'Anomali'})n_anomaly = (df_model['Anomaly'] == -1).sum()print(f"Total anomali terdeteksi: {n_anomaly} dari {len(df_model)} hari perdagangan ({n_anomaly/len(df_model)*100:.2f}%)")

In [ ]:
anomalies = df_model[df_model['Anomaly'] == -1].sort_values('Anomaly_Score')print(f"Top 10 Anomali Terkuat:")print(anomalies[['Date', 'Close', 'Log_Return', 'Residual', 'Rolling_Vol_30', 'Volume_Ratio', 'Anomaly_Score']].head(10).to_string(index=False))

## 6. Visualisasi Hasil Deteksi Anomali**Justifikasi Matematis:**Visualisasi merupakan instrumen validasi kualitatif untuk memeriksa apakah titik-titik anomali yang terdeteksi secara algoritmis berkorespondensi dengan peristiwa pasar yang diketahui (krisis finansial global 2008, pandemi COVID-19 2020, dll). Overlay anomali pada grafik harga memungkinkan evaluasi visual terhadap *precision* model — apakah alarm yang dihasilkan bersifat *actionable* dan bukan *false positive*.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))ax.plot(df_model['Date'], df_model['Close'], color='steelblue', lw=0.8, alpha=0.9, label='Harga Close BBCA')anomaly_points = df_model[df_model['Anomaly'] == -1]ax.scatter(    anomaly_points['Date'], anomaly_points['Close'],    color='red', s=40, zorder=5, alpha=0.8,    label=f'Anomali Terdeteksi (n={len(anomaly_points)})', edgecolors='darkred', linewidths=0.5)ax.set_title('Deteksi Anomali Pasar Saham BBCA — Isolation Forest', fontsize=14, fontweight='bold')ax.set_xlabel('Tanggal')ax.set_ylabel('Harga Close (IDR)')ax.legend(loc='upper left')plt.tight_layout()plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)for i, feat in enumerate(features):    axes[i].plot(df_model['Date'], df_model[feat], color='steelblue', lw=0.6, alpha=0.7)    anom = df_model[df_model['Anomaly'] == -1]    axes[i].scatter(anom['Date'], anom[feat], color='red', s=25, zorder=5, alpha=0.8)    axes[i].set_title(f'Anomali pada Fitur: {feat}')    axes[i].set_ylabel(feat)axes[2].set_xlabel('Tanggal')plt.suptitle('Distribusi Anomali per Fitur', fontsize=14, fontweight='bold', y=1.01)plt.tight_layout()plt.show()

## 7. Explainable AI — SHAP Analysis**Justifikasi Matematis:**SHAP (*SHapley Additive exPlanations*) didasarkan pada teori nilai Shapley dari *cooperative game theory*. Untuk model $ dan observasi $, kontribusi fitur $ didefinisikan sebagai:86\phi_j(f, x) = \sum_{S \subseteq N \setminus \{j\}} rac{|S|!(|N|-|S|-1)!}{|N|!} [f(S \cup \{j\}) - f(S)]86di mana $ adalah himpunan seluruh fitur dan $ adalah subset fitur tanpa fitur $. Nilai ini merepresentasikan kontribusi marjinal rata-rata fitur $ terhadap prediksi, dengan mempertimbangkan seluruh kemungkinan koalisi fitur. SHAP memenuhi properti *local accuracy*, *missingness*, dan *consistency* — menjadikannya metode interpretasi yang secara matematis *fair* dan *complete*.Analisis SHAP pada Isolation Forest memungkinkan identifikasi **mengapa** suatu hari dikategorikan sebagai anomali: apakah didominasi oleh lonjakan volatilitas, deviasi residual dari tren, atau aktivitas volume yang tidak biasa.

In [ ]:
X_df = pd.DataFrame(X_scaled, columns=features)explainer = shap.Explainer(    iso_forest.predict,    X_df,    algorithm='permutation')shap_values = explainer(X_df)

In [ ]:
plt.figure(figsize=(10, 6))shap.summary_plot(shap_values, X_df, plot_type='dot', show=False)plt.title('SHAP Summary Plot — Kontribusi Fitur terhadap Deteksi Anomali', fontsize=12)plt.tight_layout()plt.show()

In [ ]:
plt.figure(figsize=(10, 5))shap.summary_plot(shap_values, X_df, plot_type='bar', show=False)plt.title('SHAP Feature Importance (Mean |SHAP Value|)', fontsize=12)plt.tight_layout()plt.show()

In [ ]:
anomaly_indices = df_model[df_model['Anomaly'] == -1].index.tolist()if len(anomaly_indices) > 0:    idx = anomaly_indices[0]    print(f"SHAP Explanation untuk anomali terkuat (index={idx}):")    print(f"Tanggal: {df_model.loc[idx, 'Date']}")    print(f"Harga Close: {df_model.loc[idx, 'Close']}")    print()    shap.plots.waterfall(shap_values[idx], show=True)

## 8. Ringkasan & KesimpulanNotebook ini mengimplementasikan *Early Warning System* untuk deteksi anomali pasar saham BBCA dengan pendekatan hybrid:1. **Uji Asumsi Statistik** mengkonfirmasi bahwa return saham melanggar asumsi normalitas dan menunjukkan struktur autokorelasi — memvalidasi kebutuhan akan metode nonparametrik.2. **STL Decomposition** mengekstrak komponen residual sebagai sinyal deviasi dari pola normal harga.3. **Isolation Forest** mendeteksi titik-titik anomali berdasarkan kombinasi residual, volatilitas, dan volume secara nonparametrik.4. **SHAP Analysis** memberikan interpretasi transparan terhadap setiap prediksi anomali, menjawab pertanyaan *"mengapa alarm ini berbunyi?"*Pendekatan ini merepresentasikan peran Matematika Kritis dalam era digital: tidak sekadar menerima asumsi model klasik, melainkan menguji, mendekomposisi, dan menginterpretasi data secara kritis untuk menghasilkan sistem peringatan dini yang *explainable* dan *actionable*.